# Phase 5 ResNet grid — Colab execution notebook

Orchestration only (CLAUDE.md §6: "no logic in notebooks") — every training/aggregation step
below shells out to the already-committed `scripts/train_resnet_baseline.py` and
`scripts/aggregate_resnet_grid.py`. This notebook does not implement any new scientific logic;
it only sequences, resumes, and validates calls to code that already exists in `src/`/`scripts/`.

**Scope, exactly per `ACTION_PLAN.md` line 174 (Phase 5):** MONAI 3D ResNet-18 and ResNet-34,
5 patient-level CV folds (`splits/cv_folds.json`, frozen since Phase 2), 3 seeds each — 30 runs
total. Nothing else. No convergence experiment, no diagnostic run, no protocol change. The
scientific hyperparameters (`max_epochs=100`, patience 15, masked multitask BCE, AdamW,
cosine-warmup LR) come from `configs/train/default.yaml` and are never overridden here —
`--epoch-limit` only ever caps a single Colab session's wall-clock (see `docs/adr/004-colab-training-workflow.md`
decision #2); it is a different parameter from `--max-epochs` and this notebook never passes
`--max-epochs`.

**Locked test set:** `splits/test_patients.json` is never read by anything in this notebook —
`scripts/train_resnet_baseline.py` asserts no test-ID leak into any CV fold at training time, and
a fresh disjointness check runs again below before any training starts.

**Known state going in (see `PROGRESS.md`):**
- Three 5-epoch pilot runs (`resnet18_fold_0_seed{0,1,2}`) were archived to
  `experiments/_pilot_archive/` — they used `--max-epochs 5`, an incompatible schedule, and must
  never be treated as grid results.
- A `resnet18_fold_0_seed0` run was started under the real protocol (`--epoch-limit 40`, no
  `--max-epochs` override, so `max_epochs` resolved to the config default of 100) and manually
  interrupted after epoch 6, mid-validation of epoch 7. Checkpoints `epoch_0.pt`...`epoch_6.pt`
  exist; no `metrics.json` was written (the process never reached that code). This notebook does
  **not** auto-resume it — see the dedicated inspection section below.

**Safe to rerun from the top after a disconnect:** every step below either checks existing state
before acting, or is naturally idempotent (mounting Drive, `git pull`, reinstalling already-installed
packages). Training resumption is keyed off `experiments/<run_id>/metrics.json` and
`experiments/<run_id>/checkpoints/`, not off anything in-memory in this notebook.


## 1. Mount Google Drive (persistent storage for checkpoints/results across sessions)

In [ ]:
import os

# Colab's own kernel process sets MPLBACKEND to an IPython-specific inline backend
# (module://matplotlib_inline.backend_inline). subprocess.run() and `!` shell-outs below inherit
# this from os.environ by default, but that backend string is only valid inside an actual
# IPython/Jupyter process - a bare `.venv/bin/python -c ...` subprocess crashes as soon as
# anything imports matplotlib (monai does, transitively, via monai.utils.jupyter_utils).
# Overriding it once here, early, propagates to every subprocess/`!` call for the rest of this
# session - a headless backend is correct for all of them regardless.
os.environ["MPLBACKEND"] = "Agg"

from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/glioma-biomarker-data'
!mkdir -p "$DRIVE_ROOT/tumor_crop_cache" "$DRIVE_ROOT/experiments"


## 2. Clone the repo, or reuse it if this session already has it

Idempotent: if `/content/glioma-biomarker` already exists (e.g. this cell re-run after a
reconnect within the same VM), this pulls the latest `main` instead of re-cloning.


In [ ]:
from pathlib import Path

REPO = Path("/content/glioma-biomarker")

if REPO.exists():
    print(f"{REPO} already exists - pulling latest main instead of re-cloning.")
    !cd "$REPO" && git fetch origin && git checkout main && git pull origin main
else:
    !git clone https://github.com/pranathii-31/glioma-biomarker.git "$REPO"

!cd "$REPO" && git rev-parse HEAD


## 3. Install dependencies into a Python 3.10 virtualenv

`pyproject.toml` pins `requires-python = ">=3.10,<3.12"`; Colab's system Python is newer, so a
separate 3.10 venv is required for `pip install -e ".[dev]"` (which enforces that constraint) to
succeed at all — see `docs/ENVIRONMENT.md`. Idempotent: skips venv creation if `.venv` already
exists (e.g. after a reconnect within the same VM), and `pip install` itself is a no-op for
already-satisfied pins.

**Install order matters here.** `pyproject.toml` pins `torch==2.14.0` with no index annotation
(it was resolved against macOS/MPS - see `docs/ENVIRONMENT.md`). If `pip install -e ".[dev]"` runs
first, pip's resolver tries to satisfy that exact pin from the default PyPI index and Colab
reports "no matching distribution" - the CUDA (`cu121`) build of this exact version isn't published
there under a plain `pip install torch`. The fix is order, not the pin: install torch **first**,
from **plain PyPI, not the legacy `download.pytorch.org/whl/cu121` index** - that index only
serves the old `+cu121`-suffixed build scheme (versions up to roughly `2.2.x+cu121`; it does not
carry `2.14.0` at all, confirmed by the actual "no matching distribution" error against that index).
Modern PyTorch releases, `2.14.0` included, publish CUDA-enabled Linux wheels directly on plain
PyPI under the unsuffixed version string, pulling in the `nvidia-*` CUDA runtime packages as
ordinary dependencies - no special index needed. By the time `-e ".[dev]"` runs, pip finds
`torch==2.14.0` already installed and exactly satisfies the requirement without re-resolving it.
The pinned version and the CUDA requirement are both left exactly as they were - no scientific
parameter changes, and the `cuda.is_available()` check below still hard-fails rather than silently
accepting a CPU-only wheel.


In [ ]:
import subprocess

VENV = REPO / ".venv"
VENV_PY = VENV / "bin" / "python"

if not VENV_PY.exists():
    !sudo apt-get update -qq
    !sudo apt-get install -y -qq python3.10 python3.10-venv python3.10-dev
    !python3.10 -m venv "$VENV"
    !"$VENV_PY" -m pip install -q --upgrade pip
else:
    print(f"{VENV} already exists - reusing it.")

# Plain PyPI, not the legacy cu121 index - see the install-order note above. Same pin (2.14.0),
# same CUDA requirement; the cuda.is_available() check right below is what actually guards
# against silently accepting a CPU-only wheel, not the index choice.
!"$VENV_PY" -m pip install -q torch==2.14.0

torch_check = subprocess.run(
    [str(VENV_PY), "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True, text=True,
)
print(torch_check.stdout.strip() or torch_check.stderr.strip())
if "True" not in torch_check.stdout:
    raise RuntimeError(
        "torch installed but reports cuda.is_available() == False - check Runtime > Change "
        "runtime type > GPU before continuing. Not proceeding on a CPU-only torch."
    )

# pyradiomics==3.0.1's legacy setup.py imports numpy directly at build time without declaring
# it as a PEP 517 build requirement - pip's build isolation gives it a fresh, numpy-less
# environment to build in, so building it as part of a normal `-e ".[dev]"` resolve fails with
# "Getting requirements to build wheel ... error". docs/ENVIRONMENT.md's Phase 5 update already
# documents the fix: pip install it separately with --no-build-isolation, numpy already present.
# Same pre-satisfy-the-pin trick as torch above - by the time `-e ".[dev]"` runs, both numpy and
# pyradiomics already exactly match their pins and don't need to be built again.
!"$VENV_PY" -m pip install -q numpy==2.2.6
!"$VENV_PY" -m pip install -q pyradiomics==3.0.1 --no-build-isolation

# Now torch, numpy, and pyradiomics are all already installed and satisfy pyproject.toml's pins
# exactly, so this does not need to (and will not) rebuild any of them.
!cd "$REPO" && "$VENV_PY" -m pip install -q -e ".[dev]"


## 4. Verify the environment (report what's actually true - don't assume)

In [ ]:
import subprocess

version = subprocess.run([str(VENV_PY), "--version"], capture_output=True, text=True)
print("Interpreter:", VENV_PY)
print(version.stdout.strip() or version.stderr.strip())

check = subprocess.run(
    [str(VENV_PY), "-c",
     "import torch, monai, glioma; "
     "print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available()); "
     "print('monai', monai.__version__); "
     "print('glioma importable from', glioma.__file__); "
     "print('bf16 supported:', torch.cuda.is_bf16_supported() "
     "if torch.cuda.is_available() else None)"],
    capture_output=True, text=True, cwd=REPO,
)
print(check.stdout)
if check.returncode != 0:
    print(check.stderr)
    raise RuntimeError(
        "Environment verification failed - glioma/torch/monai not importable in the venv. "
        "Do not proceed to training until this is fixed."
    )


## 5-6. Verify the `tumor_crop` cache is present (count check only, not integrity verification)

**Actual Drive location, confirmed by directly listing Drive** (not the `glioma-biomarker-data/
tumor_crop_cache` path the original pilot notebook assumed - that path exists but is empty in this
Drive account; the real cache lives at `glioma-biomarker/tumor_crop_96a957c0`, found via
`find /content/drive/MyDrive -iname "tumor_crop*" -type d`):
`/content/drive/MyDrive/glioma-biomarker/tumor_crop_96a957c0`.

This cell only counts cached `.npy` files - it does **not** re-verify shape/dtype/finiteness the
way `tests/test_preprocess.py` does. Missing patients are not an error here: `GliomaVolumeDataset`
already skips-and-warns on any requested-but-uncached patient (see the
`WARNING glioma.data.dataset: N of M requested patient(s) not yet cached` lines in training output)
rather than crashing, exactly as it does locally.


In [ ]:
import subprocess

DRIVE_CACHE_DIR = Path("/content/drive/MyDrive/glioma-biomarker/tumor_crop_96a957c0")
LOCAL_PROCESSED = REPO / "data" / "processed"
LOCAL_TARGET = LOCAL_PROCESSED / DRIVE_CACHE_DIR.name
LOCAL_PROCESSED.mkdir(parents=True, exist_ok=True)

if not DRIVE_CACHE_DIR.exists():
    raise RuntimeError(
        f"{DRIVE_CACHE_DIR} not found on Drive. Confirm the cache's actual location with "
        '`!find /content/drive/MyDrive -iname "tumor_crop*" -type d` before continuing - do not '
        "guess a different path."
    )

LOCAL_TARGET.mkdir(parents=True, exist_ok=True)
drive_count = len(list(DRIVE_CACHE_DIR.glob("*.npy")))
local_count = len(list(LOCAL_TARGET.glob("*.npy")))
print(f"Drive source: {drive_count} .npy files. Local copy so far: {local_count}.")

if local_count < drive_count:
    # rsync, not cp: the Drive FUSE mount is known to drop mid-copy on Colab under sustained I/O
    # over ~3GB of many small files ("Transport endpoint is not connected"). rsync only transfers
    # files that are missing/changed, so rerunning this cell after a drop resumes instead of
    # restarting the whole copy from zero - cp -r has no such resumability.
    !which rsync > /dev/null || (apt-get update -qq && apt-get install -y -qq rsync)
    print("Copying cache from Drive (rsync - safe to rerun this cell if it's interrupted; only "
          "missing/changed files are re-transferred)...")
    !rsync -a --info=progress2 "$DRIVE_CACHE_DIR"/ "$LOCAL_TARGET"/
else:
    print(f"{LOCAL_TARGET} already has all {drive_count} files - not re-copying.")

local_count = len(list(LOCAL_TARGET.glob("*.npy")))
if local_count < drive_count:
    raise RuntimeError(
        f"Copy incomplete: {local_count}/{drive_count} .npy files present locally after rsync. "
        "The Drive FUSE mount likely dropped again - remount Drive (rerun the Drive-mount cell "
        "above, or use the Files pane's Reconnect) and rerun this cell; rsync will only transfer "
        "what's still missing, not start over."
    )
print(f"OK: all {local_count}/{drive_count} .npy files present locally.")

cache_dirs = [
    d for d in LOCAL_PROCESSED.iterdir() if d.is_dir() and d.name.startswith("tumor_crop_")
]
if not cache_dirs:
    raise RuntimeError(
        f"No tumor_crop_* cache directory found under {LOCAL_PROCESSED} - cannot train."
    )
for d in cache_dirs:
    n = len(list(d.glob("*.npy")))
    print(f"{d.name}: {n} cached .npy files (count check only - not a content/integrity check)")


## Symlink `experiments/` to Drive - persistence across a full Colab disconnect

**Added after losing the earlier pilot archive and the interrupted `resnet18_fold_0_seed0`
checkpoints**: this notebook's `experiments/` directory was plain local Colab disk with no Drive
backing, so when that session's VM was recycled, everything under it was gone - confirmed via
`find /content/drive/MyDrive` returning nothing for either. No scientific result was lost (see the
note in the pilot-archive guard cell below), but the resumability this notebook is supposed to
provide across a full disconnect (not just a cell interrupt within one live VM) requires
`experiments/` to actually live on Drive.

**Expected non-empty case, every fresh clone:** `experiments/{majority,age_only,age_sex,
radiomics_gbm}_..._2026091*` arrives from `git clone` itself - `.gitignore` only excludes
`experiments/*/checkpoints/` and `*.pt`/`*.ckpt`, not the small `metrics.json`/`config.yaml`/etc.,
so the already-committed Phase 5 cheap-baseline results (commit `7144851`) are real, tracked repo
content, not something written this session. That gets merged into Drive (never overwriting
anything already there with something different - `rsync -a`, same as the cache copy) rather than
raising, since raising on every single fresh clone would make this cell useless. A local directory
containing anything else (e.g. real per-fold ResNet checkpoints that were never intended to be
git-tracked) is still merged the same way - nothing is deleted without first being copied to Drive.


In [ ]:
import shutil

DRIVE_EXPERIMENTS = Path(DRIVE_ROOT) / "experiments"
LOCAL_EXPERIMENTS = REPO / "experiments"
DRIVE_EXPERIMENTS.mkdir(parents=True, exist_ok=True)

if LOCAL_EXPERIMENTS.is_symlink():
    print(f"{LOCAL_EXPERIMENTS} is already a symlink -> {LOCAL_EXPERIMENTS.resolve()}")
else:
    if LOCAL_EXPERIMENTS.exists() and any(LOCAL_EXPERIMENTS.iterdir()):
        print(f"{LOCAL_EXPERIMENTS} has content - merging into {DRIVE_EXPERIMENTS} before "
              "symlinking (rsync, nothing deleted until it's copied).")
        !rsync -a "$LOCAL_EXPERIMENTS"/ "$DRIVE_EXPERIMENTS"/
        shutil.rmtree(LOCAL_EXPERIMENTS)
    elif LOCAL_EXPERIMENTS.exists():
        LOCAL_EXPERIMENTS.rmdir()  # empty local dir, safe to replace with the symlink
    LOCAL_EXPERIMENTS.symlink_to(DRIVE_EXPERIMENTS, target_is_directory=True)
    print(f"Symlinked {LOCAL_EXPERIMENTS} -> {DRIVE_EXPERIMENTS} - checkpoints/results now "
          "survive a full Colab disconnect, not just an interrupted cell.")


## 7. CUDA smoke test

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

smoke = subprocess.run(
    [str(VENV_PY), "-c",
     "import torch; x = torch.randn(4, 4, device='cuda'); y = x @ x; "
     "print('CUDA tensor op OK, device:', y.device, 'peak mem MB:', "
     "torch.cuda.max_memory_allocated()/1e6)"],
    capture_output=True, text=True,
)
print(smoke.stdout)
if smoke.returncode != 0:
    print(smoke.stderr)
    raise RuntimeError("CUDA smoke test failed - check Runtime > Change runtime type > GPU.")


## 8. Grid configuration

The full required grid, exactly per `ACTION_PLAN.md` line 174 - 5 folds x 3 seeds x
{resnet18, resnet34} = 30 runs. `SESSION_EPOCH_BUDGET` is a pure execution-budget knob
(how many additional absolute epochs one training-script invocation is allowed to run before
stopping without marking convergence, per `loop.py`'s `epoch_limit` semantics - it is **not** a
scientific parameter and is never passed as `--max-epochs`). `PROTOCOL_MAX_EPOCHS` mirrors
`configs/train/default.yaml`'s `max_epochs: 100` for the compatibility checks below - it is never
passed to the training script either; leaving `--max-epochs` unset lets the script read the real
config value itself, which is the only source of truth for it.


In [ ]:
ARCHITECTURES = ["resnet18", "resnet34"]
FOLDS = [f"fold_{i}" for i in range(5)]
SEEDS = [0, 1, 2]

SESSION_EPOCH_BUDGET = 40  # epochs of additional progress per training-script invocation
# must match configs/train/default.yaml - for validation only, never passed as a flag
PROTOCOL_MAX_EPOCHS = 100

EXPERIMENTS_DIR = REPO / "experiments"
PILOT_ARCHIVE_DIR = EXPERIMENTS_DIR / "_pilot_archive"
SPLITS_DIR = REPO / "splits"

REQUIRED_RUNS = [
    (architecture, fold, seed)
    for architecture in ARCHITECTURES
    for fold in FOLDS
    for seed in SEEDS
]
print(f"{len(REQUIRED_RUNS)} required grid runs configured (expect 30).")
assert len(REQUIRED_RUNS) == 30


## Splits integrity guard

Independent re-check (mirrors `tests/test_no_leakage.py`'s logic) that the lock-box test set and
the 5 CV folds are still disjoint, read fresh from disk right before any training starts. This is
a guard against a corrupted/edited `splits/` file reaching Colab, not a re-generation of anything -
`splits/` is never written to by this notebook.


In [ ]:
import json

folds = json.loads((SPLITS_DIR / "cv_folds.json").read_text())
test_ids = set(json.loads((SPLITS_DIR / "test_patients.json").read_text()))

all_fold_ids = set()
for fold_name, ids in folds.items():
    overlap_with_test = set(ids) & test_ids
    if overlap_with_test:
        raise RuntimeError(f"LEAK: {fold_name} overlaps the locked test set: {overlap_with_test}")
    overlap_with_other_folds = set(ids) & all_fold_ids
    if overlap_with_other_folds:
        raise RuntimeError(f"LEAK: {fold_name} overlaps another fold: {overlap_with_other_folds}")
    all_fold_ids |= set(ids)

print(f"OK: {len(test_ids)} locked test patients, {len(all_fold_ids)} dev-set patients across "
      f"{len(folds)} folds, all disjoint.")


## Pilot-archive isolation guard

Checks whether the 3 archived 5-epoch pilot run directories are intact under
`experiments/_pilot_archive/`. **As of 2026-09-15, confirmed via
`find /content/drive/MyDrive -path "*_pilot_archive*"` (and `*resnet18_fold_0_seed0*`,
`*epoch_6.pt*`) returning nothing: the archive and the separately-interrupted
`resnet18_fold_0_seed0` checkpoints only ever existed on a previous Colab session's local disk and
were lost when that VM was recycled — `experiments/` had no Drive backing yet at the time (fixed
above).** This is not a scientific loss: the pilots' AUC/timing values are already recorded in
`PROGRESS.md`, and they never counted toward the required grid regardless (`--max-epochs 5`, an
incompatible schedule per `docs/adr/004-colab-training-workflow.md` decision #2) — there is
nothing to restore. The interrupted run never wrote a `metrics.json`, so nothing "reportable" was
lost there either; it simply restarts from epoch 0 the next time the grid loop reaches it.

So: an empty/missing archive here is now an **expected**, not suspicious, state — this cell warns
rather than raises for that case. It still raises if the archive directory exists but is
*incomplete* (some but not all 3 expected runs present), since that would be a genuinely odd state
worth investigating. Separately, `run_status()` below independently refuses to treat any run whose
`metrics.json` records `max_epochs != PROTOCOL_MAX_EPOCHS` as valid - so if an incompatible
5-epoch-schedule run ever reappears directly under `experiments/`, it is still caught there, not
silently aggregated, regardless of what this cell finds.


In [ ]:
PILOT_RUN_IDS = {"resnet18_fold_0_seed0", "resnet18_fold_0_seed1", "resnet18_fold_0_seed2"}
archived_present = (
    {p.name for p in PILOT_ARCHIVE_DIR.glob("*")} if PILOT_ARCHIVE_DIR.exists() else set()
)

if not archived_present:
    print(
        f"No pilot archive found under {PILOT_ARCHIVE_DIR} - expected, not an error (see markdown "
        "above: confirmed lost with the previous session's local-only VM disk, nothing to restore, "
        "nothing counted toward the grid). Proceeding."
    )
else:
    missing_from_archive = PILOT_RUN_IDS - archived_present
    if missing_from_archive:
        raise RuntimeError(
            f"Pilot archive exists under {PILOT_ARCHIVE_DIR} but is missing expected run(s): "
            f"{missing_from_archive} - a partial archive is unexpected, investigate before "
            "training."
        )
    print(
        f"OK: all 3 pilot runs remain archived under {PILOT_ARCHIVE_DIR}: "
        f"{sorted(archived_present)}"
    )


## Inspect the interrupted `resnet18_fold_0_seed0` run before anything resumes it

Per-instruction: do **not** blindly resume or discard this run. `loop.py`'s checkpoint format does
not store `max_epochs` (it's a caller-supplied argument each invocation, not part of saved state -
see `docs/adr/004-colab-training-workflow.md` decision #2), so this cell can only report what *is*
recorded in the checkpoint. Whether the run is actually safe to resume depends on knowing no
`--max-epochs` override was ever passed to it - this notebook itself never passes one, but it
cannot verify what happened before this notebook existed.


In [ ]:
import torch


def inspect_checkpoint(architecture: str, fold: str, seed: int) -> dict | None:
    run_id = f"{architecture}_{fold}_seed{seed}"
    run_dir = EXPERIMENTS_DIR / run_id
    if PILOT_ARCHIVE_DIR in run_dir.parents or run_dir == PILOT_ARCHIVE_DIR:
        raise RuntimeError(f"Refusing to inspect {run_dir} - it is inside the pilot archive.")

    checkpoint_dir = run_dir / "checkpoints"
    checkpoints = list(checkpoint_dir.glob("epoch_*.pt")) if checkpoint_dir.exists() else []
    if not checkpoints:
        print(f"{run_id}: no checkpoints found - nothing to inspect.")
        return None
    latest = max(checkpoints, key=lambda p: int(p.stem.removeprefix("epoch_")))

    ckpt = torch.load(latest, map_location="cpu")
    metrics_path = run_dir / "metrics.json"
    print(f"{run_id}: latest checkpoint = {latest.name}")
    print(f"  last completed epoch (0-indexed): {ckpt['epoch']}")
    print(f"  best_auc so far: {ckpt['best_auc']:.4f}")
    print(f"  epochs_without_improvement: {ckpt['epochs_without_improvement']}")
    print(f"  epoch_log entries: {len(ckpt['epoch_log'])}")
    print(f"  metrics.json present (run previously finished cleanly): {metrics_path.exists()}")
    print()
    print("  CANNOT VERIFY from the checkpoint file alone: what --max-epochs value (if any) was "
          f"used to produce it. This notebook never passes --max-epochs (always leaves it at the "
          f"configs/train/default.yaml default, currently {PROTOCOL_MAX_EPOCHS}). If you have any "
          "reason to believe a --max-epochs override was used for this specific checkpoint outside "
          "this notebook, archive it (same as the 5-epoch pilots) instead of resuming it.")
    return ckpt

_resnet18_fold_0_seed0_checkpoint = inspect_checkpoint("resnet18", "fold_0", 0)


## Manually confirm which unverified checkpoints are safe to resume

Only add a run_id here **after** reading its `inspect_checkpoint(...)` output above and confirming
you know it was never launched with a `--max-epochs` override. The grid loop below refuses to
auto-resume any checkpoint-only (no `metrics.json`) run that is not in this set - it raises instead
of guessing.


In [ ]:
CONFIRMED_COMPATIBLE_RUNS: set[str] = {
    # "resnet18_fold_0_seed0",  # uncomment only after reviewing the inspection cell's output above
}


## 9-13. Run-status detection, checkpoint/resume, and the 30-run grid

`run_status` inspects `experiments/<run_id>/` and returns one of:
- `"complete"` - has `metrics.json`, and either `converged` or reached `PROTOCOL_MAX_EPOCHS` - skip.
- `"resume_with_metrics"` - has `metrics.json` but `ran_out_of_epoch_budget` - resume normally.
- `"resume_unverified"` - has checkpoints but no `metrics.json` (e.g. an interrupted run) - only
  auto-resumed if its run_id is in `CONFIRMED_COMPATIBLE_RUNS`, else raises.
- `"not_started"` - nothing yet - starts fresh (no `--resume` flag, `--epoch-limit` = the session budget).

Every invocation always passes `--architecture --fold --seed`, `--epoch-limit` (grown from the
last known absolute epoch as needed - see the module docstring in `src/glioma/train/loop.py` for
why this can't be a fixed number across resumes), and **never** `--max-epochs`. A non-zero
return code from the training script stops the whole loop immediately (`check=True`) rather than
silently continuing to the next run.


In [ ]:
def run_status(architecture: str, fold: str, seed: int):
    run_id = f"{architecture}_{fold}_seed{seed}"
    run_dir = EXPERIMENTS_DIR / run_id
    metrics_path = run_dir / "metrics.json"
    checkpoint_dir = run_dir / "checkpoints"

    if metrics_path.exists():
        m = json.loads(metrics_path.read_text())
        if m.get("max_epochs") != PROTOCOL_MAX_EPOCHS:
            raise RuntimeError(
                f"{run_id}: metrics.json records max_epochs={m.get('max_epochs')!r}, expected "
                f"{PROTOCOL_MAX_EPOCHS} - incompatible schedule. Archive this run's directory "
                "(same as the 5-epoch pilots) rather than resuming or aggregating it."
            )
        if m.get("converged") or (m.get("stopped_epoch", -1) + 1 >= PROTOCOL_MAX_EPOCHS):
            return "complete", m
        return "resume_with_metrics", m

    checkpoints = list(checkpoint_dir.glob("epoch_*.pt")) if checkpoint_dir.exists() else []
    if checkpoints:
        if run_id not in CONFIRMED_COMPATIBLE_RUNS:
            raise RuntimeError(
                f"{run_id}: has checkpoints but no metrics.json, and is not in "
                "CONFIRMED_COMPATIBLE_RUNS. Run inspect_checkpoint(...) on it, review the output, "
                "and add it to CONFIRMED_COMPATIBLE_RUNS above before rerunning this cell - this "
                "notebook refuses to blindly resume an unverified checkpoint."
            )
        return "resume_unverified", None

    return "not_started", None


def latest_absolute_epoch(architecture: str, fold: str, seed: int) -> int:
    checkpoint_dir = EXPERIMENTS_DIR / f"{architecture}_{fold}_seed{seed}" / "checkpoints"
    checkpoints = list(checkpoint_dir.glob("epoch_*.pt"))
    return max(int(p.stem.removeprefix("epoch_")) for p in checkpoints)


In [ ]:
import time


def train_one_run(architecture: str, fold: str, seed: int) -> dict:
    run_id = f"{architecture}_{fold}_seed{seed}"
    run_dir = EXPERIMENTS_DIR / run_id
    print(f"=== {run_id} ===")

    status, existing_metrics = run_status(architecture, fold, seed)

    if status == "complete":
        m = existing_metrics
        print(f"  Already complete - skipping. "
              f"epochs={m['stopped_epoch'] + 1}, converged={m['converged']}, "
              f"IDH AUC={m['val_auc']['idh']:.4f}, MGMT AUC={m['val_auc']['mgmt']:.4f}")
        return {"run_id": run_id, "status": "complete", "resumed": False, **m}

    resume = status in ("resume_with_metrics", "resume_unverified")
    while True:
        current_epoch = latest_absolute_epoch(architecture, fold, seed) if resume else -1
        epoch_limit = current_epoch + 1 + SESSION_EPOCH_BUDGET

        cmd = [
            str(VENV_PY), "scripts/train_resnet_baseline.py",
            "--architecture", architecture, "--fold", fold, "--seed", str(seed),
            "--epoch-limit", str(epoch_limit),
        ]
        if resume:
            cmd.append("--resume")

        print(f"  Running: {' '.join(cmd)}")
        t0 = time.time()
        result = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
        elapsed = time.time() - t0
        print(result.stdout[-3000:])
        if result.returncode != 0:
            print(result.stderr[-3000:])
            raise RuntimeError(
                f"{run_id} failed (exit {result.returncode}) - stopping the grid here. "
                "Inspect the output above before rerunning this cell."
            )

        metrics_path = run_dir / "metrics.json"
        if not metrics_path.exists():
            raise RuntimeError(
                f"{run_id}: training script exited 0 but wrote no metrics.json - unexpected, "
                "stopping rather than guessing at its state."
            )
        m = json.loads(metrics_path.read_text())
        print(f"  run_dir={run_dir}  resumed={resume}  "
              f"epochs_completed={m['stopped_epoch'] + 1}  converged={m['converged']}  "
              f"IDH AUC={m['val_auc']['idh']:.4f}  MGMT AUC={m['val_auc']['mgmt']:.4f}  "
              f"elapsed_this_invocation={elapsed:.1f}s  "
              f"checkpoint_dir={run_dir / 'checkpoints'}")

        if m["converged"] or (m["stopped_epoch"] + 1 >= PROTOCOL_MAX_EPOCHS):
            return {"run_id": run_id, "status": "complete", "resumed": resume, **m}
        if not m.get("ran_out_of_epoch_budget"):
            raise RuntimeError(
                f"{run_id}: stopped without converging and without ran_out_of_epoch_budget set - "
                "unexpected state, stopping rather than looping indefinitely."
            )
        resume = True  # continue the inner loop with a grown epoch_limit


## Run the grid

Sequential, one run at a time, 30 runs total. Each run's inner loop (above) keeps resuming within
this same cell until that run converges or reaches `PROTOCOL_MAX_EPOCHS`, then moves to the next
run. If this cell or the whole session dies partway through, rerunning it from the top picks up
exactly where it left off via `run_status` - nothing above this point needs to be redone except
re-running cells 1-7 to restore the session's mounts/venv/GPU.


In [ ]:
grid_summary = []
for architecture, fold, seed in REQUIRED_RUNS:
    grid_summary.append(train_one_run(architecture, fold, seed))


## Grid summary

In [ ]:
completed = [r for r in grid_summary if r["status"] == "complete"]
print(f"Required runs: {len(REQUIRED_RUNS)}")
print(f"Completed: {len(completed)}")
print(f"Not completed: {len(REQUIRED_RUNS) - len(completed)}")

missing_or_invalid = []
for architecture, fold, seed in REQUIRED_RUNS:
    run_id = f"{architecture}_{fold}_seed{seed}"
    match = next((r for r in grid_summary if r["run_id"] == run_id), None)
    if match is None or match["status"] != "complete":
        missing_or_invalid.append(run_id)

if missing_or_invalid:
    print(f"\nMissing/incomplete runs ({len(missing_or_invalid)}): {missing_or_invalid}")
    print("\nAggregation is NOT safe to run yet.")
else:
    print("\nAll 30 required runs complete. Aggregation is safe to run.")


## 14-15. Aggregation and `results/baselines.md` (only once the summary above says it's safe)

Refuses to proceed (raises) unless all 30 required runs are confirmed complete - do not skip the
summary cell above or edit around this check.


In [ ]:
if missing_or_invalid:
    raise RuntimeError(
        f"Refusing to aggregate: {len(missing_or_invalid)} run(s) not complete: "
        f"{missing_or_invalid}"
    )

for architecture in ARCHITECTURES:
    for seed in SEEDS:
        cmd = [str(VENV_PY), "scripts/aggregate_resnet_grid.py",
               "--architecture", architecture, "--seed", str(seed)]
        print(f"Running: {' '.join(cmd)}")
        result = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError(f"Aggregation failed for architecture={architecture} seed={seed}")


In [ ]:
result = subprocess.run(
    [str(VENV_PY), "scripts/build_results_table.py"], cwd=REPO, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("scripts/build_results_table.py failed")

print((REPO / "results" / "baselines.md").read_text())


## Bring results back to the local repo

`experiments/` here is a symlink into Drive if you've pointed it there, or plain local disk
otherwise - either way, **do not `git add` from inside this notebook.** On your local machine:
sync the aggregated `experiments/*/metrics.json` (+ `config.yaml`/`git_commit.txt`/
`splits_hash.txt`/`env.txt`, never `checkpoints/`) down into the local repo's `experiments/`,
rerun `make lint && make test`, regenerate `results/baselines.md`, and commit from there - keeps
commit authorship, pre-commit hooks, and the git safety protocol on the machine that has them
configured. Do not commit anything from this notebook.
